# MSTR TPC-DS Schema Import

Imports all `mstr_view` BigQuery views as logical tables into the MSTR project `mstr-tpc`.

**Approach A** — mstrio-py (`list_warehouse_tables` → `add_to_project`)  
**Approach B** — MSTR REST API using pre-built JSON files in `config/osi_import_json/`

References: [mstrio-py table_mgmt.py](https://github.com/MicroStrategy/mstrio-py/blob/master/code_snippets/table_mgmt.py)

## Imports & config

In [107]:
import json
import yaml
import os
import glob
import time
from mstrio.api import tables
import glob, json, os
from pprint import pprint
from mstrio.modeling.schema.fact import list_facts
from mstr_robotics.mstr_classes import get_conn
from mstrio.modeling import SchemaManagement, SchemaUpdateType
from mstrio.modeling.schema.helpers import ObjectSubType, SchemaObjectReference
from mstrio.modeling.schema.table import (
    list_warehouse_tables,
    list_logical_tables,
    LogicalTable,
)


In [142]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
#object ids are maintained in ..\config\jupyter_objects_d.json
import json
with open("..\\config\\jupyter_objects_d.json", "r") as openfile:
    jupyter_objects_d = json.load(openfile)
nb_d = jupyter_objects_d["jup_mstr_tpcds_schema_import"]
PROJECT_ID   = nb_d["project_id"]   # mstr-tpc project
FACTS_FOLDER_ID = nb_d["folders"]["FACTS_FOLDER_ID"]
DATASOURCE_ID=nb_d["misc"]["DATASOURCE_ID"]
METRICS_FOLDER_ID = nb_d["folders"]["METRICS_FOLDER_ID"]
ATTR_FOLDER_ID = nb_d["folders"]["ATTR_FOLDER_ID"]

DTYPE_MAP_FILE = str(OSI_FILES / "mstr" / "db_enums" / "db_data_type_form_type.yaml")
JSON_DIR       = str(OSI_FILES)         # pre-built REST API payloads
TABLE_DIR      = ".\\mstr\\mstr_tutorial\\table_def"
FACT_DIR       = ".\\mstr\\mstr_tutorial\\fact_def"
ATTR_DIR       = ".\\mstr\\mstr_tutorial\\attribute_def"
PC_DIR         = ".\\mstr\\mstr_tutorial\\attribute_def\\parentChild"
METRIC_DIR     = ".\\mstr\\mstr_tutorial\\metric_def"
OSI_YAML       = ".\\mstr\\mstr_tutorial\\osi_import_schema.yaml"

os.chdir(JSON_DIR)

with open(OSI_YAML, "r", encoding="utf-8") as f:
    osi_model = yaml.safe_load(f)

PC_DIR        = os.path.join(ATTR_DIR, "parentChild")

os.makedirs(ATTR_DIR, exist_ok=True)
os.makedirs(PC_DIR, exist_ok=True)

## Connect to MSTR project

In [143]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
user_conf=USER_CONFIG
with open(user_conf, 'r') as openfile:
    user_d = json.load(openfile)
conn_params=user_d["conn_params"]
print(conn_params)
conn = get_conn(
    base_url=conn_params["base_url"],
    username=conn_params["username"],
    password=conn_params["password"],
    project_id=PROJECT_ID
)
conn.headers["Content-type"] = "application/json"
print(f"Connected → project_id={conn.project_id}")


{'username': 'Administrator', 'password': '[REMOVED-PASSWORD]', 'base_url': 'http://217.154.213.84:8080/MicroStrategyLibrary/api'}
Connection to Strategy One Intelligence Server has been established.
Connected → project_id=35249BC4472353A88620DCAF854F304A


In [20]:
os.makedirs(TABLE_DIR, exist_ok=True)

datasets = osi_model["semantic_model"][0]["datasets"]
print(f"Found {len(datasets)} table(s) in OSI model\n")

for ds in datasets:
    table_name = ds["name"]
    source = ds.get("source", "")
    parts = source.split(".")
    # source format: <db>.<schema>.<table>
    namespace = parts[-2] if len(parts) >= 2 else ""
    namespace=""
    physical_name = parts[-1] if parts else table_name

    payload = {
        "information": {
            "name": table_name
        },
        "primaryDataSource": {
            "objectId": DATASOURCE_ID
        },
        "physicalTable": {
            "tableName": physical_name,
            "namespace": namespace
        }
    }

    out_path = os.path.join(TABLE_DIR, f"{table_name}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"  {table_name}.json")

print(f"\nDone — {len(datasets)} JSON files written to: {TABLE_DIR}")

Found 15 table(s) in OSI model

  order_fact.json
  order_detail.json
  lu_day.json
  lu_customer.json
  lu_item.json
  lu_subcateg.json
  lu_category.json
  rel_cat_item.json
  lu_country.json
  lu_region.json
  lu_pymt_type.json
  lu_employee.json
  lu_promotion.json
  lu_shipper.json
  lu_brand.json

Done — 15 JSON files written to: .\mstr\mstr_tutorial\table_def


## REST API: POST /api/model/tables using JSON files

In [21]:

json_files = sorted(glob.glob(os.path.join(TABLE_DIR, "*.json")))
print(f"Found {len(json_files)} files\n")

results = {"ok": [], "error": []}

for fp in json_files:
    tbl = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, "r", encoding="utf-8") as f:
        d = json.load(f)
    try:
        resp = tables.post_table(connection=conn, data=d)
        if resp.status_code in (200, 201):
            print(f"  ✓  {tbl}  →  id={resp.json().get('id','?')}")
            results["ok"].append(tbl)
        else:
            print(f"  ✗  {tbl}  HTTP {resp.status_code}: {resp.text[:200]}")
            results["error"].append({"table": tbl, "error": resp.text[:200]})
    except Exception as e:
        print(f"  ✗  {tbl}: {e}")
        results["error"].append({"table": tbl, "error": str(e)})

print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")

Found 15 files

  ✓  lu_brand  →  id=632E1D4625094EAC969BF3B1A2CC2D30
  ✓  lu_category  →  id=55759DE553084B8B80900C252E5CB4A1
  ✓  lu_country  →  id=E5C2D9BC66DC497CB784ED872962B734
  ✓  lu_customer  →  id=57D8F31A00344E3A87A54595C035D800
  ✓  lu_day  →  id=3851EA7C4C3C417287471A6BBB226C15
  ✓  lu_employee  →  id=F17FC655FEE74DDBB245228492D3FE26
  ✓  lu_item  →  id=C9EF04506A914F7B927ED207D3EB11BF
  ✓  lu_promotion  →  id=EB7C8F9FB63D4270AB6CAE58E28EC94E
  ✓  lu_pymt_type  →  id=9E8D5074AEC14BA4852311515DED0EBB
  ✓  lu_region  →  id=728096909DED4C2D9E4C8CE878DAE8BB
  ✓  lu_shipper  →  id=07F0149930A04591A5F655FBFFD99E20
  ✓  lu_subcateg  →  id=DC037EAE269A439B9D9EE8FEFAF8480D
  ✓  order_detail  →  id=8C607FB0F8CE4DE98A05D28B945DD5BB
  ✓  order_fact  →  id=A42BE9FA6DD343E583AB44823DD07301
  ✓  rel_cat_item  →  id=C875DF3510BD4DF9B4C6E6BC4CC28CC1

Done — 15 ok, 0 errors


## Reload schema after import

In [22]:
def reload_schema(conn, project_id: str):
    """
    Triggers a schema reload so imported tables are visible
    in reports and dashboards.
    """
    schema_mgr = SchemaManagement(connection=conn, project_id=project_id)
    task = schema_mgr.reload(update_types=[SchemaUpdateType.LOGICAL_SIZE])
    print(f"Schema reload triggered: {task}")
    return task


# Uncomment after import is complete
# reload_schema(conn, PROJECT_ID)
print("reload_schema() ready — run after import completes")

reload_schema() ready — run after import completes


## Facts — generate JSON files from OSI metrics

In [23]:
import re
import yaml

def get_table_id(conn, table_name: str, _cache: dict = {}) -> str:
    """
    Returns the MSTR objectId for a logical table by name.
    Results are cached after the first call so list_logical_tables
    is only called once per session.
    """
    if not _cache:
        tables = list_logical_tables(conn)
        _cache.update({t.name: t.id for t in tables})
        print(f"[get_table_id] cached {len(_cache)} logical tables")

    if table_name not in _cache:
        raise KeyError(
            f"Logical table '{table_name}' not found in project. "
            f"Available: {sorted(_cache)}"
        )
    return _cache[table_name]


def parse_ansi_expr(expr_str: str):
    """Parses 'AGG(table.column)' → (table_name, column_name)."""
    m = re.match(r"\w+\((\w+)\.(\w+)\)", expr_str.strip())
    if not m:
        raise ValueError(f"Cannot parse OSI expression: {expr_str!r}")
    return m.group(1), m.group(2)


def build_fact_payload(conn, metric: dict, folder_id: str) -> dict:
    ansi_expr = next(
        d["expression"] for d in metric["expression"]["dialects"]
        if d["dialect"] == "ANSI_SQL"
    )
    table_name, col_name = parse_ansi_expr(ansi_expr)
    tbl_id = get_table_id(conn, table_name)
    return {
        "information": {
            "name": metric["name"],
            "subType": "fact",
            "destinationFolderId": folder_id,
        },
        "dataType": {"type": "float", "precision": 8, "scale": -2147483648},
        "expressions": [{
            "expression": {"tokens": [{"value": col_name}]},
            "tables": [{
                "objectId": tbl_id,
                "subType": "logical_table",
                "name": table_name,
            }],
        }],
    }


def generate_fact_json_files(
    conn,
    osi_yaml_path: str,
    fact_dir: str,
    folder_id: str = "__FACTS_FOLDER_ID__",
) -> list:
    """
    Reads OSI YAML metrics and writes one JSON file per metric into fact_dir.
    Table IDs are resolved live from MSTR via get_table_id().
    Returns list of (metric_name, file_path) tuples.
    """
    with open(osi_yaml_path, "r", encoding="utf-8") as f:
        osi = yaml.safe_load(f)

    metrics = osi["semantic_model"][0]["metrics"]
    os.makedirs(fact_dir, exist_ok=True)
    results = []

    for metric in metrics:
        try:
            payload = build_fact_payload(conn, metric, folder_id)
            fp = os.path.join(fact_dir, f"{metric['name']}.json")
            with open(fp, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)
            results.append((metric["name"], fp))
        except Exception as e:
            print(f"  ✗  {metric['name']}  ({e})")
            continue
        tbl  = payload["expressions"][0]["tables"][0]["name"]
        col  = payload["expressions"][0]["expression"]["tokens"][0]["value"]
        tbl_id = payload["expressions"][0]["tables"][0]["objectId"]
        print(f"  ✓  {metric['name']}  ({tbl}.{col}  tbl_id={tbl_id})")

    print(f"\nGenerated {len(results)} fact JSON files in: {fact_dir}")
    return results


fact_files = generate_fact_json_files(conn, OSI_YAML, FACT_DIR)


[get_table_id] cached 15 logical tables
  ✓  gross_sales  (order_fact.gross_dollar_sales  tbl_id=A42BE9FA6DD343E583AB44823DD07301)
  ✓  net_sales  (order_fact.order_amt  tbl_id=A42BE9FA6DD343E583AB44823DD07301)
  ✓  total_cost  (order_fact.order_cost  tbl_id=A42BE9FA6DD343E583AB44823DD07301)
  ✓  gross_margin  (order_fact.order_amt  tbl_id=A42BE9FA6DD343E583AB44823DD07301)
  ✗  gross_margin_pct  (Cannot parse OSI expression: 'ROUND(100.0 * (SUM(order_fact.order_amt) - SUM(order_fact.order_cost)) / NULLIF(SUM(order_fact.order_amt), 0), 2)')
  ✓  units_sold  (order_fact.qty_sold  tbl_id=A42BE9FA6DD343E583AB44823DD07301)
  ✗  order_count  (Cannot parse OSI expression: 'COUNT(DISTINCT order_fact.order_id)')
  ✓  avg_order_value  (order_fact.gross_dollar_sales  tbl_id=A42BE9FA6DD343E583AB44823DD07301)
  ✗  customer_count  (Cannot parse OSI expression: 'COUNT(DISTINCT order_fact.customer_id)')
  ✓  customer_lifetime_value  (order_fact.gross_dollar_sales  tbl_id=A42BE9FA6DD343E583AB44823DD073

## Facts — patch folder ID into JSON files, then create in MSTR

In [24]:
from mstrio.api import facts


def patch_fact_folder_id(fact_dir: str, folder_id: str) -> None:
    """Replaces __FACTS_FOLDER_ID__ in all fact JSON files with the real folder_id."""
    patched = 0
    for fp in sorted(glob.glob(os.path.join(fact_dir, "*.json"))):
        with open(fp, "r", encoding="utf-8") as f:
            body = json.load(f)
        body["information"]["destinationFolderId"] = folder_id
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(body, f, indent=2)
        patched += 1
    print(f"Patched destinationFolderId={folder_id} in {patched} files")


def create_facts_from_json(conn, fact_dir: str) -> dict:
    """
    Loops through all JSON files in fact_dir and creates each as a MSTR fact
    via POST /api/model/facts (changeset managed automatically by mstrio-py).
    Returns dict with ok / error lists.
    """
    json_files = sorted(glob.glob(os.path.join(fact_dir, "*.json")))
    print(f"Found {len(json_files)} fact JSON files\n")

    results = {"ok": [], "error": []}

    for fp in json_files:
        fact_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            d = json.load(f)
        try:
            resp = facts.create_fact(connection=conn, body=d)
            if resp.status_code in (200, 201):
                fact_id = resp.json().get("id", "?")
                print(f"  ✓  {fact_name}  →  id={fact_id}")
                results["ok"].append({"name": fact_name, "id": fact_id})
            else:
                print(f"  ✗  {fact_name}  HTTP {resp.status_code}: {resp.text[:300]}")
                results["error"].append({"name": fact_name, "error": resp.text[:300]})
        except Exception as e:
            print(f"  ✗  {fact_name}: {e}")
            results["error"].append({"name": fact_name, "error": str(e)})

    print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results


# 1. patch folder ID
patch_fact_folder_id(FACT_DIR, FACTS_FOLDER_ID)

# 2. create facts
fact_results = create_facts_from_json(conn, FACT_DIR)


Patched destinationFolderId=D4BB03B950C14CB0946D64FFC136FEF5 in 7 files
Found 7 fact JSON files

  ✓  avg_order_value  →  id=9A98802BCEDD4EADAB53269294B5FDF2
  ✓  customer_lifetime_value  →  id=B959EB47D27B4B0DB3B90795DA095C12
  ✓  gross_margin  →  id=4DA38F45E2004859988C0B34F3466102
  ✓  gross_sales  →  id=22A68F5E03D44237BB634ED7E73E806F
  ✓  net_sales  →  id=869F98F7D38B48509ACE2D05FC7AD47E
  ✓  total_cost  →  id=1F30F85123BD4B65978C46F95D0BC6A9
  ✓  units_sold  →  id=618AF5F3983249D68B98B0045DC25F39

Done — 7 ok, 0 errors


## Metrics — generate JSON files from OSI metrics

In [25]:
#METRIC_DIR = r"..\config\osi_create_metric"

# MSTR system-constant object ID for the Sum aggregation function
SUM_FUNCTION_ID = nb_d["misc"]["SUM_FUNCTION_ID"]

# metrics whose underlying measure is a count/quantity → integer data type
INTEGER_METRICS = {"store_quantity", "inventory_on_hand"}


def get_fact_id(conn, fact_name: str, _cache: dict = {}) -> str:
    """
    Returns the MSTR objectId for a fact by name.
    Results are cached after the first call so list_facts is only called once.
    """
    if not _cache:
        all_facts = list_facts(connection=conn)
        _cache.update({f.name: f.id for f in all_facts})
        print(f"[get_fact_id] cached {len(_cache)} facts")

    if fact_name not in _cache:
        raise KeyError(
            f"Fact '{fact_name}' not found in project. "
            f"Available: {sorted(_cache)}"
        )
    return _cache[fact_name]


def build_metric_payload(conn, metric_name: str, folder_id: str) -> dict:
    """
    Builds the REST API JSON body for a simple SUM(fact) metric.
    Expression tokens: Sum function → ( → fact object_reference → ) → end_of_text
    """
    fact_id = get_fact_id(conn, metric_name)
    dtype = "integer" if metric_name in INTEGER_METRICS else "float"
    precision, scale = (10, 0) if dtype == "integer" else (8, -2147483648)

    return {
        "information": {
            "name": metric_name,
            "subType": "metric",
            "destinationFolderId": folder_id,
        },
        "expression": {
            "tokens": [
                {
                    "value": "Sum",
                    "type": "function",
                    "target": {
                        "objectId": SUM_FUNCTION_ID,
                        "subType": "function",
                        "name": "Sum",
                    },
                },
                {"value": "(", "type": "character"},
                {
                    "value": metric_name,
                    "type": "object_reference",
                    "target": {
                        "objectId": fact_id,
                        "subType": "fact",
                        "name": metric_name,
                    },
                },
                {"value": ")", "type": "character"},
                {"value": "", "type": "end_of_text"},
            ]
        },
        "dataType": {"type": dtype, "precision": precision, "scale": scale},
    }


def generate_metric_json_files(
    conn,
    metrics: list,
    metric_dir: str,
    folder_id: str = "__METRICS_FOLDER_ID__",
) -> list:
    """
    Accepts the already-parsed OSI metrics list (no file I/O),
    resolves fact IDs from MSTR, and writes one JSON file per metric.
    Returns list of (metric_name, file_path) tuples.
    """
    os.makedirs(metric_dir, exist_ok=True)
    results = []

    for m in metrics:
        try:
            name = m["name"]
            payload = build_metric_payload(conn, name, folder_id)
            fp = os.path.join(metric_dir, f"{name}.json")
            with open(fp, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)
        except Exception as e:
            print(f"  ✗  {name}  ({e})")
            continue
        results.append((name, fp))
        fact_id = payload["expression"]["tokens"][2]["target"]["objectId"]
        dtype   = payload["dataType"]["type"]
        print(f"  ✓  {name}  (fact_id={fact_id}  dtype={dtype})")

    print(f"\nGenerated {len(results)} metric JSON files in: {metric_dir}")
    return results


# Parse OSI YAML once — reuse for both facts and metrics sections
with open(OSI_YAML, "r", encoding="utf-8") as f:
    osi_model = yaml.safe_load(f)

osi_metrics = osi_model["semantic_model"][0]["metrics"]
metric_files = generate_metric_json_files(conn, osi_metrics, METRIC_DIR)

`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
[get_fact_id] cached 7 facts
  ✓  gross_sales  (fact_id=22A68F5E03D44237BB634ED7E73E806F  dtype=float)
  ✓  net_sales  (fact_id=869F98F7D38B48509ACE2D05FC7AD47E  dtype=float)
  ✓  total_cost  (fact_id=1F30F85123BD4B65978C46F95D0BC6A9  dtype=float)
  ✓  gross_margin  (fact_id=4DA38F45E2004859988C0B34F3466102  dtype=float)
  ✗  gross_margin_pct  ("Fact 'gross_margin_pct' not found in project. Available: ['avg_order_value', 'customer_lifetime_value', 'gross_margin', 'gross_sales', 'net_sales', 'total_cost', 'units_sold']")
  ✓  units_sold  (fact_id=618AF5F3983249D68B98B0045DC25F39  dtype=float)
  ✗  order_count  ("Fact 'order_count' not found in project. Available: ['avg_order_value', 'customer_lifetime_value', 'gross_margin', 'gross_sales', 'net_sales', 'total_cost', 'units_sold']")
  ✓  avg_order_value  (fact_id=9A98802BCEDD4EADAB53269294B5FDF2  dtype=float)
  ✗  customer_count  ("Fact '

## Metrics — patch folder ID into JSON files, then create in MSTR

In [26]:
from mstrio.api import metrics as metrics_api


def patch_metric_folder_id(metric_dir: str, folder_id: str) -> None:
    """Writes the real destinationFolderId into all metric JSON files."""
    patched = 0
    for fp in sorted(glob.glob(os.path.join(metric_dir, "*.json"))):
        with open(fp, "r", encoding="utf-8") as f:
            body = json.load(f)
        body["information"]["destinationFolderId"] = folder_id
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(body, f, indent=2)
        patched += 1
    print(f"Patched destinationFolderId={folder_id} in {patched} metric files")


def create_metrics_from_json(conn, metric_dir: str) -> dict:
    """
    Loops through all JSON files in metric_dir and creates each as a MSTR
    metric via POST /api/model/metrics.
    Changeset lifecycle is managed automatically by mstrio-py.
    Returns dict with ok / error lists.
    """
    json_files = sorted(glob.glob(os.path.join(metric_dir, "*.json")))
    print(f"Found {len(json_files)} metric JSON files\n")

    results = {"ok": [], "error": []}

    for fp in json_files:
        metric_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            d = json.load(f)
        try:
            resp = metrics_api.create_metric(connection=conn, body=d)
            if resp.status_code in (200, 201):
                metric_id = resp.json().get("id", "?")
                print(f"  ✓  {metric_name}  →  id={metric_id}")
                results["ok"].append({"name": metric_name, "id": metric_id})
            else:
                print(f"  ✗  {metric_name}  HTTP {resp.status_code}: {resp.text[:300]}")
                results["error"].append({"name": metric_name, "error": resp.text[:300]})
        except Exception as e:
            print(f"  ✗  {metric_name}: {e}")
            results["error"].append({"name": metric_name, "error": str(e)})

    print(f"\nDone — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results


# 1. patch real folder ID into all metric JSON files
patch_metric_folder_id(METRIC_DIR, METRICS_FOLDER_ID)

# 2. create metrics in MSTR
metric_results = create_metrics_from_json(conn, METRIC_DIR)


Patched destinationFolderId=E0CCB9CF22104A489CBE78D974AFD19E in 7 metric files
Found 7 metric JSON files

  ✓  avg_order_value  →  id=F46FC34E12044D61BB57E6377E659724
  ✓  customer_lifetime_value  →  id=1C57AE0A51864C09A5E110F6DB26A46D
  ✓  gross_margin  →  id=EE74D6D8C6994B2BA88D57BA55A3707E
  ✓  gross_sales  →  id=D20AC411D28E4DDC85C268563521477A
  ✓  net_sales  →  id=7A8AD6C00E0F4968A72BFC58647AD1C5
  ✓  total_cost  →  id=7777C1787ED14AF489F3ACEDA55A2728
  ✓  units_sold  →  id=006581D09CAD4E77A298BA96F305AA9D

Done — 7 ok, 0 errors


## Create Attribute JSON files

In [110]:
# Create Attribute JSON files

# ── Load data-type mapping ────────────────────────────────────────────────────
with open(DTYPE_MAP_FILE, "r", encoding="utf-8") as _f:
    _dtype_cfg = yaml.safe_load(_f)

# Flat lookup keyed by UPPERCASE db type name
_TYPE_MAP         = {k.upper(): v for k, v in _dtype_cfg["type_map"].items()}
# Ordered list of (type_map_key, [suffixes]) — first match wins
_SUFFIX_HEURISTICS = list(_dtype_cfg["suffix_heuristics"].items())
_DTYPE_DEFAULT    = _dtype_cfg["default"]

print(f"Loaded {len(_TYPE_MAP)} db→mstr type mappings from {DTYPE_MAP_FILE}")

# osi_model was loaded in the metrics cell — reused here
datasets      = osi_model["semantic_model"][0]["datasets"]
osi_relations = osi_model["semantic_model"][0]["relationships"]


# ── helpers ───────────────────────────────────────────────────────────────────

def get_column_id(conn, table_name: str, column_name: str, _cache: dict = {}) -> str:
    """
    Returns the MSTR column objectId for a given table + column name.
    Calls LogicalTable.list_columns() once per table and caches the result.
    """
    if table_name not in _cache:
        table_id = get_table_id(conn, table_name)
        lt = LogicalTable(conn, id=table_id)
        cols = lt.list_columns()
        _cache[table_name] = {c.name: c.id for c in cols}
        print(f"[get_column_id] cached {len(_cache[table_name])} columns for '{table_name}'")

    col_map = _cache[table_name]
    if column_name not in col_map:
        raise KeyError(
            f"Column '{column_name}' not found in '{table_name}'. "
            f"Available: {sorted(col_map)}"
        )
    return col_map[column_name]


def _col_dtype(col_name: str, db_type: str | None = None) -> tuple[dict, str]:
    """
    Returns (dataType dict, displayFormat string) for an MSTR attribute form.

    Resolution order:
      1. db_type provided → case-insensitive lookup in _TYPE_MAP
      2. no db_type (or not found) → suffix heuristics from _SUFFIX_HEURISTICS
      3. fallback → _DTYPE_DEFAULT

    All type values come from config/osi/db_enums/db_data_type_form_type.yaml.
    """
    # 1 — direct db_type lookup
    if db_type:
        entry = _TYPE_MAP.get(db_type.upper())
        if entry:
            return (
                {"type": entry["mstr_type"], "precision": entry["precision"], "scale": entry["scale"]},
                entry["display_format"],
            )

    # 2 — suffix heuristics (checked in YAML declaration order)
    lc = col_name.lower()
    for type_key, suffixes in _SUFFIX_HEURISTICS:
        if any(lc.endswith(s) for s in suffixes):
            entry = _TYPE_MAP.get(type_key, _DTYPE_DEFAULT)
            return (
                {"type": entry["mstr_type"], "precision": entry["precision"], "scale": entry["scale"]},
                entry["display_format"],
            )

    # 3 — fallback
    return (
        {"type": _DTYPE_DEFAULT["mstr_type"], "precision": _DTYPE_DEFAULT["precision"], "scale": _DTYPE_DEFAULT["scale"]},
        _DTYPE_DEFAULT["display_format"],
    )


def build_attribute_payload(conn, dataset: dict, relations: list, folder_id: str) -> dict | None:
    """
    Builds the REST API JSON body for an MSTR attribute with full form coverage.

    ID form (category='ID'):
        • Expression 1: PK column on the lookup table itself
        • Expressions 2..N: FK columns on every fact/dim table that references
          this attribute (OSI relationships where this table is the 'to'/PK side).

    Descriptive forms (one per non-PK column in dataset["fields"]):
        • category='DESC' for the first descriptive column, 'NONE' for the rest
        • Single expression on the lookup table only
        • Data type resolved via _col_dtype() → db_data_type_form_type.yaml
    """
    table_name = dataset["name"]
    pk_cols    = dataset.get("primary_key", [])
    if not pk_cols:
        return None

    pk_col   = pk_cols[0]
    table_id = get_table_id(conn, table_name)
    col_id   = get_column_id(conn, table_name, pk_col)

    lookup_table_ref = {
        "objectId": table_id,
        "subType":  "logical_table",
        "name":     table_name,
    }

    # ── ID form: PK on lookup table + FK on every referencing table ──────────
    # MSTR reuses the same column objectId for same-named columns across tables.
    # When the FK column shares an objectId with an existing expression, add the
    # FK table to that expression's tables array (the "Source tables" checkboxes
    # in the MSTR editor) instead of creating a duplicate expression.
    col_id_to_expr_idx = {col_id: 0}
    id_expressions = [
        {
            "expression": {"tokens": [
                {"value": pk_col, "type": "column_reference",
                 "target": {"objectId": col_id, "subType": "column", "name": pk_col}},
                {"value": "", "type": "end_of_text"},
            ]},
            "tables": [lookup_table_ref],
        }
    ]
    for rel in relations:
        if rel["to"] != table_name:
            continue
        fk_table = rel["from"]
        fk_cols  = rel.get("from_columns", [])
        if not fk_cols:
            continue
        fk_col = fk_cols[0]
        try:
            fk_table_id  = get_table_id(conn, fk_table)
            fk_col_id    = get_column_id(conn, fk_table, fk_col)
            fk_table_ref = {"objectId": fk_table_id, "subType": "logical_table", "name": fk_table}
            if fk_col_id in col_id_to_expr_idx:
                # Same column object → extend the existing expression's tables list
                id_expressions[col_id_to_expr_idx[fk_col_id]]["tables"].append(fk_table_ref)
                print(f"  + mapped {fk_table}.{fk_col} to existing expression (shared column objectId)")
            else:
                # Different column object (e.g. differently-named FK) → new expression
                col_id_to_expr_idx[fk_col_id] = len(id_expressions)
                id_expressions.append({
                    "expression": {"tokens": [
                        {"value": fk_col, "type": "column_reference",
                         "target": {"objectId": fk_col_id, "subType": "column", "name": fk_col}},
                        {"value": "", "type": "end_of_text"},
                    ]},
                    "tables": [fk_table_ref],
                })
        except KeyError as e:
            print(f"  ⚠  skipping FK expression {fk_table}.{fk_col} → {table_name}: {e}")

    forms = [
        {
            "name":          "ID",
            "alias":         pk_col,
            "category":      "ID",
            "displayFormat": "number",
            "dataType":      {"type": "integer", "precision": 8, "scale": 0},
            "expressions":   id_expressions,
            "lookupTable":   lookup_table_ref,
        }
    ]

    # ── Descriptive forms: one per non-PK column from OSI fields ─────────────
    non_pk_fields = [f for f in dataset.get("fields", []) if f["name"] != pk_col]
    desc_assigned = False

    for field in non_pk_fields:
        col_name = field["name"]
        # Use db_type from OSI model if present (field.get("type")), else suffix heuristic
        db_type = field.get("type")
        try:
            col_id_f = get_column_id(conn, table_name, col_name)
        except KeyError:
            print(f"  ⚠  '{col_name}' not in MSTR schema for '{table_name}' — skipping form")
            continue

        dtype, dfmt = _col_dtype(col_name, db_type=db_type)
        category    = "DESC" if not desc_assigned else "NONE"
        desc_assigned = True

        forms.append({
            "name":          col_name,
            "alias":         col_name,
            "category":      category,
            "displayFormat": dfmt,
            "dataType":      dtype,
            "expressions": [
                {
                    "expression": {"tokens": [
                        {"value": col_name, "type": "column_reference",
                         "target": {"objectId": col_id_f, "subType": "column", "name": col_name}},
                        {"value": "", "type": "end_of_text"},
                    ]},
                    "tables": [lookup_table_ref],
                }
            ],
            "lookupTable": lookup_table_ref,
        })

    browse_displays = [{"name": "ID"}]
    if non_pk_fields:
        browse_displays.append({"name": non_pk_fields[0]["name"]})

    return {
        "information": {
            "name":                table_name,
            "subType":             "attribute",
            "destinationFolderId": folder_id,
        },
        "forms":   forms,
        "keyForm": {"name": "ID"},
        "displays": {
            "reportDisplays": [{"name": "ID"}],
            "browseDisplays":  browse_displays,
        },
        "attributeLookupTable": lookup_table_ref,
    }


# ── generate attribute JSON files (one per dimension) ─────────────────────────

def generate_attribute_json_files(
    conn,
    datasets: list,
    relations: list,
    attr_dir: str,
    fact_tables: set,
    folder_id: str = "__ATTR_FOLDER_ID__",
) -> list:
    results = []
    for ds in datasets:
        name = ds["name"]
        if name in fact_tables:
            continue
        payload = build_attribute_payload(conn, ds, relations, folder_id)
        if payload is None:
            print(f"  ⚠  {name} — skipped (no primary_key in OSI)")
            continue
        fp = os.path.join(attr_dir, f"{name}.json")
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        results.append((name, fp))
        n_forms  = len(payload["forms"])
        n_id_exp = len(payload["forms"][0]["expressions"])
        print(f"  ✓  {name}  (forms={n_forms}  id_expressions={n_id_exp})")
    print(f"\nGenerated {len(results)} attribute JSON files in: {attr_dir}")
    return results


def get_fact_tables_from_osi(osi_model: dict) -> set:
    """
    Derives the set of fact table names from the OSI model by inspecting
    which tables are referenced in metric (fact) expressions.
    Reuses parse_ansi_expr so no separate classification file is needed.
    """
    fact_tables = set()
    for metric in osi_model["semantic_model"][0].get("metrics", []):
        for dialect in metric.get("expression", {}).get("dialects", []):
            if dialect["dialect"] == "ANSI_SQL":
                try:
                    table_name, _ = parse_ansi_expr(dialect["expression"])
                    fact_tables.add(table_name)
                except ValueError:
                    pass
    print(f"Identified {len(fact_tables)} fact table(s) from OSI model: {sorted(fact_tables)}")
    return fact_tables


# ── run ───────────────────────────────────────────────────────────────────────
fact_tables = get_fact_tables_from_osi(osi_model)

print("=== Attribute JSON files ===")
attr_files = generate_attribute_json_files(
    conn, datasets, osi_relations, ATTR_DIR,
    fact_tables=fact_tables,
)


Loaded 44 db→mstr type mappings from C:\coding\Python_environments\OSI_Files\mstr\db_enums\db_data_type_form_type.yaml
Identified 1 fact table(s) from OSI model: ['order_fact']
=== Attribute JSON files ===
`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
LogicalTable object named: 'order_detail' with ID: '8C607FB0F8CE4DE98A05D28B945DD5BB'
[get_column_id] cached 10 columns for 'order_detail'
  ⚠  'line_revenue' not in MSTR schema for 'order_detail' — skipping form
  ⚠  'line_cost' not in MSTR schema for 'order_detail' — skipping form
  ⚠  'line_margin' not in MSTR schema for 'order_detail' — skipping form
  ✓  order_detail  (forms=10  id_expressions=1)
`project_id` and `project_name` were not provided. Project from `connection` object is used instead.
LogicalTable object named: 'lu_day' with ID: '3851EA7C4C3C417287471A6BBB226C15'
[get_column_id] cached 8 columns for 'lu_day'
`project_id` and `project_name` were not provided. Project fr

## Attributes — patch folder ID into JSON files, then create in MSTR

In [80]:
from mstrio.api import attributes as attr_api

def patch_attr_folder_id(attr_dir: str, folder_id: str) -> None:
    """Writes the real destinationFolderId into all attribute JSON files."""
    patched = 0
    for fp in sorted(glob.glob(os.path.join(attr_dir, "*.json"))):
        with open(fp, "r", encoding="utf-8") as f:
            body = json.load(f)
        body["information"]["destinationFolderId"] = folder_id
        with open(fp, "w", encoding="utf-8") as f:
            json.dump(body, f, indent=2)
        patched += 1
    print(f"Patched destinationFolderId={folder_id} in {patched} attribute files")


def create_attributes_from_json(conn, attr_dir: str) -> dict:
    """
    Loops through all JSON files directly in attr_dir (not the parentChild/
    sub-folder) and creates each as a MSTR attribute via POST /api/model/attributes.
    Changeset lifecycle is managed automatically by mstrio-py.
    Returns dict with ok / error / skipped lists.
    """
    json_files = sorted(glob.glob(os.path.join(attr_dir, "*.json")))
    print(f"Found {len(json_files)} attribute JSON files\n")

    results = {"ok": [], "error": [], "skipped": []}

    for fp in json_files:
        attr_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            d = json.load(f)
        n_exp = len(d.get("forms", [{}])[0].get("expressions", []))
        try:
            resp = attr_api.create_attribute(connection=conn, body=d)
            if resp.status_code in (200, 201):
                attr_id = resp.json().get("id", "?")
                print(f"  ✓  {attr_name}  →  id={attr_id}  (expressions={n_exp})")
                results["ok"].append({"name": attr_name, "id": attr_id})
            else:
                err_text = resp.text
                # 8004ccfc = duplicate name in folder → attribute already exists
                if "8004ccfc" in err_text:
                    print(f"  ~  {attr_name}  already exists — skipped")
                    results["skipped"].append(attr_name)
                else:
                    print(f"  ✗  {attr_name}  HTTP {resp.status_code}: {err_text[:300]}")
                    results["error"].append({"name": attr_name, "error": err_text[:300]})
        except Exception as e:
            err_str = str(e)
            if "8004ccfc" in err_str:
                print(f"  ~  {attr_name}  already exists — skipped")
                results["skipped"].append(attr_name)
            else:
                print(f"  ✗  {attr_name}: {err_str}")
                results["error"].append({"name": attr_name, "error": err_str})

    print(f"\nDone — {len(results['ok'])} created, {len(results['skipped'])} skipped (already exist), {len(results['error'])} errors")
    return results


# 1. patch real folder ID
print("=== Patching folder ID ===")
patch_attr_folder_id(ATTR_DIR, ATTR_FOLDER_ID)

# 2. create attributes in MSTR
print("\n=== Creating attributes in MSTR ===")
attr_results = create_attributes_from_json(conn, ATTR_DIR)

=== Patching folder ID ===
Patched destinationFolderId=6F55FB47F9974EABA18CB0C5FF46785C in 14 attribute files

=== Creating attributes in MSTR ===
Found 14 attribute JSON files

  ✓  lu_brand  →  id=52FAA284467B42A981A651CDCD7CF67D  (expressions=1)
  ✓  lu_category  →  id=B2ECFAC60FA44A07BB63EFF4B5AFC7E2  (expressions=2)
  ✓  lu_country  →  id=896455BA4B6C4D299303A604B8B9A39F  (expressions=1)
  ✓  lu_customer  →  id=94ABC9D10BA54E79BF80BCCDA5600A91  (expressions=1)
  ✓  lu_day  →  id=49F07E800C524269AF83AACDE11F4F1E  (expressions=2)
  ✓  lu_employee  →  id=88911C908FD44EB3A7B98C8BFEDEA52E  (expressions=1)
  ✓  lu_item  →  id=1D43B44257574512BC4F125757EEEA68  (expressions=1)
  ✓  lu_promotion  →  id=E0AF45FAD86145C786488BC3E3D5D6AF  (expressions=1)
  ✓  lu_pymt_type  →  id=04F2257C58C74EC2BE18BE3804CB0A82  (expressions=1)
  ✓  lu_region  →  id=9BD640E5493F49548904E81C70F1CC10  (expressions=1)
  ✓  lu_shipper  →  id=7B6F06C0E0FB4766B2AE8B0C6D5FEF92  (expressions=1)
  ✓  lu_subcateg  →  i

In [111]:
# ── generate parentChild JSON files (dim-to-dim relationships only) ───────────

from mstrio.modeling.schema.attribute.attribute import Attribute
from mstrio.modeling.schema.attribute import RelationshipType
from mstrio.modeling.schema.helpers import SchemaObjectReference, ObjectSubType


def generate_parentchild_json_files(
    conn,
    osi_relations: list,
    fact_tables: set,
    pc_dir: str,
) -> list:
    """
    Generates one JSON file per dim→dim relationship in the format expected by
    the MSTR REST API (POST /api/attributes/{id}/relationships).

    objectIds are resolved directly via get_attribute_id() and get_table_id().
    The join table is always the child (FK) table as defined in the OSI model.

    JSON shape per file:
        {
          "child":             { "objectId": "...", "subType": "attribute",     "name": "..." },
          "parent":            { "objectId": "...", "subType": "attribute",     "name": "..." },
          "relationshipTable": { "objectId": "...", "subType": "logical_table", "name": "..." },
          "relationshipType":  "one_to_many"
        }
    """
    os.makedirs(pc_dir, exist_ok=True)
    results = []

    for rel in osi_relations:
        child_name  = rel["from"]   # FK side
        parent_name = rel["to"]     # PK side
        join_table  = rel["from"]   # FK table hosts the join column

        if child_name in fact_tables or parent_name in fact_tables:
            continue

        try:
            child_id      = get_attribute_id(conn, child_name)
            parent_id     = get_attribute_id(conn, parent_name)
            join_table_id = get_table_id(conn, join_table)

            payload = {
                "child": {
                    "objectId": child_id,
                    "subType":  "attribute",
                    "name":     child_name,
                },
                "parent": {
                    "objectId": parent_id,
                    "subType":  "attribute",
                    "name":     parent_name,
                },
                "relationshipTable": {
                    "objectId": join_table_id,
                    "subType":  "logical_table",
                    "name":     join_table,
                },
                "relationshipType": "one_to_many",
            }

            fp = os.path.join(pc_dir, f"{rel['name']}.json")
            with open(fp, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)
            results.append(rel["name"])
            print(f"  ✓  parent={parent_name} ({parent_id})  ←→  child={child_name} ({child_id})  via {join_table}")

        except Exception as e:
            print(f"  ✗  {rel['name']}: {e}")

    print(f"\nGenerated {len(results)} parentChild JSON files in: {pc_dir}")
    return results


def apply_parentchild_json_files(conn, pc_dir: str) -> dict:
    """
    Reads every JSON file from pc_dir and applies the parent→child relationship
    in MSTR via parent_attr.add_child().

    Uses list_attributes() and list_logical_tables() to resolve live objects
    directly — avoids a second per-object fetch (Attribute(conn, id=...)) which
    fails when the modeling API context differs from the object management API.
    """
    json_files = sorted(glob.glob(os.path.join(pc_dir, "*.json")))
    print(f"Found {len(json_files)} parentChild JSON file(s) in {pc_dir}")

    # build name → object maps once (avoids N+1 API calls)
    attr_map  = {a.name: a for a in list_attributes(connection=conn)}
    table_map = {t.name: t for t in list_logical_tables(conn)}
    print(f"Loaded {len(attr_map)} attributes, {len(table_map)} tables from MSTR")

    results = {"ok": [], "error": []}

    for fp in json_files:
        rel_name = os.path.splitext(os.path.basename(fp))[0]
        with open(fp, "r", encoding="utf-8") as f:
            rel = json.load(f)

        parent_name = rel["parent"]["name"]
        child_name  = rel["child"]["name"]
        tbl_name    = rel["relationshipTable"]["name"]

        try:
            parent_attr = attr_map[parent_name]
            child_obj   = attr_map[child_name]
            tbl_obj     = table_map[tbl_name]

            child_ref = SchemaObjectReference(
                object_id=child_obj.id,
                sub_type=ObjectSubType.ATTRIBUTE,
                name=child_name,
            )
            table_ref = SchemaObjectReference(
                object_id=tbl_obj.id,
                sub_type=ObjectSubType.LOGICAL_TABLE,
                name=tbl_name,
            )

            parent_attr.add_child(
                child=child_ref,
                relationship_type=RelationshipType.ONE_TO_MANY,
                table=table_ref,
            )
            print(f"  ✓  {parent_name}  ←[1:N]→  {child_name}  (join: {tbl_name})")
            results["ok"].append(rel_name)

        except KeyError as e:
            print(f"  ✗  {rel_name}: object not found — {e}")
            results["error"].append({"name": rel_name, "error": str(e)})
        except Exception as e:
            print(f"  ✗  {rel_name}: {e}")
            results["error"].append({"name": rel_name, "error": str(e)})

    print(f"Done — {len(results['ok'])} ok, {len(results['error'])} errors")
    return results

# ── run ───────────────────────────────────────────────────────────────────────
print("=== Generating parentChild JSON files ===")
pc_files = generate_parentchild_json_files(
    conn, osi_relations, fact_tables=fact_tables, pc_dir=PC_DIR)

print("\n=== Applying parentChild relationships in MSTR ===")
#pc_results = apply_parentchild_json_files(conn, PC_DIR)


=== Generating parentChild JSON files ===
  ✓  parent=lu_item (95EF605A4FAC4D6B98CA45FC995FC905)  ←→  child=order_detail (B7038E8DA86D49CC8051222B917CEA44)  via order_detail
  ✓  parent=lu_customer (65A1D7C2E9CE49FBB5E6B37855260F20)  ←→  child=order_detail (B7038E8DA86D49CC8051222B917CEA44)  via order_detail
  ✓  parent=lu_promotion (20ED1CE8F7FD4BA69D671091B454C3C5)  ←→  child=order_detail (B7038E8DA86D49CC8051222B917CEA44)  via order_detail
  ✓  parent=lu_employee (CF0FED87997241B0BE671FC699477657)  ←→  child=order_detail (B7038E8DA86D49CC8051222B917CEA44)  via order_detail
  ✓  parent=lu_day (7A824A48C3BA4639BCAEE45B8D23C64E)  ←→  child=order_detail (B7038E8DA86D49CC8051222B917CEA44)  via order_detail
  ✓  parent=lu_subcateg (F5F2C12EE73F451A82D53E2AB5A382F5)  ←→  child=lu_item (95EF605A4FAC4D6B98CA45FC995FC905)  via lu_item
  ✓  parent=lu_category (4AF7DC4F2E684666A02EB007D1D66856)  ←→  child=lu_subcateg (F5F2C12EE73F451A82D53E2AB5A382F5)  via lu_subcateg
  ✓  parent=lu_category (4

In [145]:
conn.select_project(PROJECT_ID)
ATTRIBUTE_ID=nb_d["misc"]["ATTRIBUTE_ID"]

parent_attr = Attribute(connection=conn, id=ATTRIBUTE_ID)
child_ref = SchemaObjectReference(
    object_id=nb_d["misc"]["child_attribute_id"],
    sub_type=ObjectSubType.ATTRIBUTE,
    name="lu_item",
)
table_ref = SchemaObjectReference(
    object_id=nb_d["misc"]["child_table_id"],
    sub_type=ObjectSubType.LOGICAL_TABLE,
    name="lu_item",
)

parent_attr.add_child(
    child=child_ref,
    relationship_type=RelationshipType.ONE_TO_MANY,
    table=table_ref,
)

Attribute object named: 'lu_brand' with ID: '52FAA284467B42A981A651CDCD7CF67D'
Attribute 'lu_brand' has been modified on the server. Your changes are saved locally.


In [ ]:
def create_changeset(conn) -> str:
    """Creates a schema-edit changeset and returns its ID."""
    resp = conn.post(
        f"{conn.base_url}/api/model/changesets",
        headers={
            "X-MSTR-AuthToken": conn.headers["X-MSTR-AuthToken"],
            "X-MSTR-ProjectID": conn.project_id
        },
        json={"schemaEdit": True},
    )
    resp.raise_for_status()
    changeset_id = resp.json()["id"]
    print(f"Changeset created: {changeset_id}")
    return changeset_id

changeset_id=create_changeset(conn=conn)